### Chain Using LangGraph
In this section we will see how we can build a simple chain using Langgraph that uses 4 important concepts

- How to use chat messages as our graph state
- How to use chat models in graph nodes
- How to bind tools to our LLM in chat models
- How to execute the tools call in our graph nodes 

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

#### How to use chat messages as our graph state
##### Messages

We can use messages which can be used to capture different roles within a conversation.
LangChain has various message types including HumanMessage, AIMessage, SystemMessage and ToolMessage.
These represent a message from the user, from chat model, for the chat model to instruct behavior, and from a tool call.

Every message have these important components.

- content - content of the message
- name - Specify the name of author
- response_metadata - optionally, a dict of metadata (e.g., often populated by model provider for AIMessages)

In [2]:
from langchain_core.messages import AIMessage,HumanMessage
from pprint import pprint

messages=[AIMessage(content=f"Please tell me how can I help",name="LLMModel")]
messages.append(HumanMessage(content=f"I want to learn coding",name="Chirag"))
messages.append(AIMessage(content=f"Which programming language you want to learn",name="LLMModel"))
messages.append(HumanMessage(content=f"I want to learn python programming language",name="Chirag"))

for message in messages:
    message.pretty_print()

================================== Ai Message ==================================
Name: LLMModel

Please tell me how can I help
================================ Human Message =================================
Name: Chirag

I want to learn coding
================================== Ai Message ==================================
Name: LLMModel

Which programming language you want to learn
================================ Human Message =================================
Name: Chirag

I want to learn python programming language


### Chat Models

We can use the sequence of message as input with the chatmodels using LLM's and OPENAI.

In [3]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-120b")
result=llm.invoke(messages)
print(result)

content='### 🎯 Goal: Get comfortable writing, reading, and debugging Python code\n\nBelow is a **step‑by‑step roadmap** you can follow at your own pace, plus **free resources**, **practice ideas**, and **tips** to keep you motivated.\n\n---\n\n## 1️⃣ Set Up Your Development Environment (≈ 30\u202fmin)\n\n| Option | Why it’s good | Quick setup steps |\n|--------|---------------|-------------------|\n| **VS Code** (free, cross‑platform) | Powerful extensions, integrated terminal, great for beginners | 1. Download & install from <https://code.visualstudio.com/> <br>2. Install the **Python** extension (Microsoft) <br>3. Install Python (see next row) |\n| **PyCharm Community Edition** | IDE with many built‑in tools | Download from <https://www.jetbrains.com/pycharm/download/> |\n| **Online editors** (e.g., Replit, Google Colab, Jupyter Notebook) | No install needed, great for quick experiments | Open <https://replit.com/> or <https://colab.research.google.com/> and create a new Python file/

In [4]:
result.response_metadata

{'token_usage': {'completion_tokens': 3072,
  'prompt_tokens': 113,
  'total_tokens': 3185,
  'completion_time': 6.402855204,
  'completion_tokens_details': {'reasoning_tokens': 34},
  'prompt_time': 0.004249582,
  'prompt_tokens_details': None,
  'queue_time': 0.317143237,
  'total_time': 6.407104786},
 'model_name': 'openai/gpt-oss-120b',
 'system_fingerprint': 'fp_4200b3f836',
 'service_tier': 'on_demand',
 'finish_reason': 'length',
 'logprobs': None,
 'model_provider': 'groq'}

### Tools
Tools can be integrated with the LLM models to interact with external systems. External systems can be API's, third party tools.

Whenever a query is asked the model can choose to call the tool and this query is based on the 
natural language input and this will return an output that matches the tool's schema

In [5]:
def add(a:int,b:int)-> int:
    return a+b

In [6]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002BC64CA0D70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002BC64CA1A90>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
### Binding tool with llm
a=2
b=2
llm_with_tools=llm.bind_tools([add])

tool_call=llm_with_tools.invoke([HumanMessage(content=f"What is 2 plus 2",name="Krish")])

In [11]:
tool_call.tool_calls

[]

### Using messages as state

In [12]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class State(TypedDict):
    message:list[AnyMessage]

#### Reducers
Now, we have a minor problem!

As we discussed, each node will return a new value for our state key messages.

But, this new value will override the prior messages value.

As our graph runs, we want to append messages to our messages state key.

We can use reducer functions to address this.

Reducers allow us to specify how state updates are performed.

If no reducer function is specified, then it is assumed that updates to the key should override it as we saw before.

But, to append messages, we can use the pre-built add_messages reducer.

This ensures that any messages are appended to the existing list of messages.

We simply need to annotate our messages key with the add_messages reducer function as metadata.

In [13]:
from langgraph.graph.message import add_messages
from typing import Annotated
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

### Reducers with add_messages

In [14]:
initial_messages=[AIMessage(content=f"Please tell me how can I help",name="LLMModel")]
initial_messages.append(HumanMessage(content=f"I want to learn coding",name="Krish"))
initial_messages

[AIMessage(content='Please tell me how can I help', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I want to learn coding', additional_kwargs={}, response_metadata={}, name='Krish')]

In [16]:
ai_message=AIMessage(content=f"Which programming language you want to learn",name="LLMModel")
ai_message

AIMessage(content='Which programming language you want to learn', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[])

In [17]:
### Reducers add_messages is to append instead of override
add_messages(initial_messages,ai_message)

[AIMessage(content='Please tell me how can I help', additional_kwargs={}, response_metadata={}, name='LLMModel', id='b7c3cae3-2f58-4650-8e4f-65c5d58c2c1b', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I want to learn coding', additional_kwargs={}, response_metadata={}, name='Krish', id='e417b6c6-4944-4452-ad7b-afd941621483'),
 AIMessage(content='Which programming language you want to learn', additional_kwargs={}, response_metadata={}, name='LLMModel', id='a6c9d53d-7cde-415e-b974-0112f54b663a', tool_calls=[], invalid_tool_calls=[])]